In [32]:
import os
import pandas as pd
import numpy as np
from app_config import PROJ_ROOT, DATA_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob
from models.DeepConvLST_toys import DeepConvLSTM, HARDataset, collate_fn, create_weighted_sampler
import optuna
import optuna.visualization as vis
import train_toys
#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\pdd_data'
print(path)

C:\codes\HumanActivityRecognition\data\pdd_data


In [33]:
#trovo tutti i file che corrispondono a "BE*" nella cartella path
#e li stampo a schermo
files = glob.glob(os.path.join(path, "*_BE*.csv"))
print("Files:", files)
#lista per salvare utenti prima del merge 
kid_be= []
#lista per salvare utenti dopo il merge
kid_be_no_null = []
#per ogni file nella lista files
#leggo il file e stampo le dimensioni del dataframe, le colonne, il conteggio delle attività e il conteggio dei giocattoli
for file in files:
    print(f"Processing: {file}")
    df = pd.read_csv(file)
    
    print("Original shape:", df.shape)
    kid_be.append(df['kid_id'].unique())
    df = df[df['action_id'] != 0] #filtro le righe con action_id non nullo
    print("Filtered shape:", df.shape)
    kid_be_no_null.append(df['kid_id'].unique())

    print("Columns:", df.columns)
    print("Action counts:\n", df['action'].value_counts())
    print("Toy counts:\n", df['toy_id'].value_counts())
    print("="*50)  # Separatore tra i file

#mi stampo gli utenti prima di fare il merge e dopo il merge
print("Users before merge:", kid_be)
print("Users after merge:", kid_be_no_null)


Files: ['C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3002_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3002_BE2.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3003_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3007_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3008_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3009_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3010_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3011_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3013_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3017_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3018_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3019_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3020_BE1.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3023_BE1.csv', 'C:\\codes\\HumanActivityR

C:\Users\carol\AppData\Local\Temp\ipykernel_5204\2113086673.py:13: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Original shape: (211480, 25)
Filtered shape: (22178, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
Action counts:
 action
prova a incastrare         12926
tiene in mano               2649
appoggia                    1516
impila                      1297
gira                        1250
mette in fila               1139
sposta posto                 720
afferra                      224
porta dietro alla testa      199
sposta                       137
lascia cadere                121
Name: count, dtype: int64
Toy counts:
 toy_id
BE1    22178
Name: count, dtype: int64
Processing: C:\codes\HumanActivityRecognition\data\pdd_data\3007_BE1.csv


C:\Users\carol\AppData\Local\Temp\ipykernel_5204\2113086673.py:13: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Original shape: (203832, 25)
Filtered shape: (10416, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
Action counts:
 action
tiene in mano              6745
porta dietro alla testa    1942
avvicina                    839
sposta posto                619
lascia cadere               271
Name: count, dtype: int64
Toy counts:
 toy_id
BE1    10416
Name: count, dtype: int64
Processing: C:\codes\HumanActivityRecognition\data\pdd_data\3008_BE1.csv


...

Original shape: (62710, 25)
Filtered shape: (4817, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
Action counts:
 action
tiene in mano         2319
prova a incastrare    1355
impila                 917
solleva                200
lascia cadere           26
Name: count, dtype: int64
Toy counts:
 toy_id
BE1    4817
Name: count, dtype: int64
Processing: C:\codes\HumanActivityRecognition\data\pdd_data\3018_BE1.csv
Original shape: (95092, 25)
Filtered shape: (0, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_ti

C:\Users\carol\AppData\Local\Temp\ipykernel_5204\2113086673.py:13: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Original shape: (178124, 25)
Filtered shape: (192, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
Action counts:
 action
sposta    192
Name: count, dtype: int64
Toy counts:
 toy_id
BE1    192
Name: count, dtype: int64
Processing: C:\codes\HumanActivityRecognition\data\pdd_data\3023_BE1.csv


C:\Users\carol\AppData\Local\Temp\ipykernel_5204\2113086673.py:13: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Original shape: (215469, 25)
Filtered shape: (124, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
Action counts:
 action
sposta    124
Name: count, dtype: int64
Toy counts:
 toy_id
BE1    124
Name: count, dtype: int64
Processing: C:\codes\HumanActivityRecognition\data\pdd_data\df_BE12_non_null.csv
Original shape: (64480, 25)
Filtered shape: (64480, 25)
Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'res

In [34]:
#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path

df_list_be = [] #lista vuota per appendere i dataframe con attività non nulla
for file in os.listdir(path):
    if file.endswith('.csv'):
        #leggo solo i file che dopo l'undescore ha BE*.csv
        if file.split('_')[-1].startswith('BE') and file.endswith('.csv'): #controllo che il file termini con .csv
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_be.append(df_temp)

df_be = pd.concat(df_list_be)
print("Dimensioni del df_be con tutte le attività non nulle")
print(df_be.shape)
print(df_be.columns)
print(df_be['action'].value_counts())

#salvo il dataframe
df_be.to_csv(os.path.join(path,'df_BE_non_null.csv'),index=False)

            

C:\Users\carol\AppData\Local\Temp\ipykernel_5204\344198626.py:11: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp=pd.read_csv(os.path.join(path,file))
C:\Users\carol\AppData\Local\Temp\ipykernel_5204\344198626.py:11: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp=pd.read_csv(os.path.join(path,file))
C:\Users\carol\AppData\Local\Temp\ipykernel_5204\344198626.py:11: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp=pd.read_csv(os.path.join(path,file))
C:\Users\carol\AppData\Local\Temp\ipykernel_5204\344198626.py:11: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp=pd.read_csv(os.path.join(path,file))
C:\Users\carol\AppData\Local\Temp\ipykernel_5204\344198626.py:11: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or

Dimensioni del df_be con tutte le attività non nulle
(64480, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
action
tiene in mano              19059
prova a incastrare         14281
sposta posto                8849
appoggia                    3514
impila                      2495
porta dietro alla testa     2141
trascina                    2040
porge                       1997
afferra                     1912
sposta                      1594
gira                        1502
mette in fila               1139
avvicina                     839
lancia                       710
lascia cadere                597
rovescia                     596
solleva                

In [35]:

#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_be['action_id'].unique():
    df_action = df_be[df_be["action_id"] == action_id] #filtro il dataframe in base all'attività
    print(f"Dimensioni del dataframe df_be_action_{action_id}")
    print(df_action.shape) #stampo le dimensioni del dataframe
    print(df_action.columns) #stampo le colonne del dataframe
    print(df_action['action'].value_counts()) #stampo il conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_be_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    print(f"Salvato il dataframe df_be_action_{action_id}.csv")



Dimensioni del dataframe df_be_action_36
(1139, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
action
mette in fila    1139
Name: count, dtype: int64
Salvato il dataframe df_be_action_36.csv
Dimensioni del dataframe df_be_action_4
(3514, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
action
appoggia    3514
Name: count, dtype: int

In [36]:
#applico sliding window con la funzion process_csv
#definisco i parametri
nb_sensor_channels = 13
sliding_window_length = 100
sliding_window_step = 20

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al toy palla
#e poi concateno tutto in un unica x e y 

X= []
Y= []

for file in os.listdir(path):
    if file.endswith('.csv') and file.split('_')[1] == 'be':
        file_path = os.path.join(path, file)
        print(file_path)
        X_windows, Y_windows = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)
        X.append(X_windows)
        Y.append(Y_windows)

# Concatenate all the windows into a single array
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)

#NUMERO TOTALE DI FINESTRE PER LA PALLA
print("Numero totale di finestre per il giocattolo be:")
print(X.shape)
print(Y.shape)

#verifca
print(Y)


C:\codes\HumanActivityRecognition\data\pdd_data\df_be_action_10.csv
Colonne nel dataset: ['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X', 'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X', 'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id', 'toy_id', 'communication', 'social_interaction', 'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E']
Feature selezionate: ['Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X', 'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X', 'Mag_Y', 'Mag_Z', 'kid_id']
Shape di X_data: (19059, 13)
Shape di Y_data: (19059,)
Shape di X_windows: (948, 100, 13)
Shape di Y_windows: (948, 1)
C:\codes\HumanActivityRecognition\data\pdd_data\df_be_action_11.csv
Colonne nel dataset: ['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X', 'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X', 'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id', 'toy

In [37]:
# Dizionario per contare le finestre per ogni azione per ogni bambino
kid_action_counts = {}

# Itero su ogni finestra in X e Y
for i in range(len(X)):  # Per ogni finestra
    kid_id = X[i, 0, -1]  # L'ID del bambino corrispondente alla finestra (ultima colonna)
    action = Y[i, 0]  # L'azione corrispondente alla finestra

    if kid_id not in kid_action_counts:  # Se il bambino non è ancora nel dizionario
        kid_action_counts[kid_id] = {}  # Inizializza un dizionario per le azioni

    if action not in kid_action_counts[kid_id]:  # Se l'azione non è ancora presente
        kid_action_counts[kid_id][action] = 0  # Inizializza il conteggio a zero
    
    kid_action_counts[kid_id][action] += 1  # Incrementa il conteggio della finestra

# Stampa i risultati
for kid_id, actions in kid_action_counts.items():
    print(f"Kid ID {kid_id}:")
    for action, count in actions.items():
        print(f"  Azione {action}: {count} finestre")


Kid ID 3003.0:
  Azione 10: 133 finestre
  Azione 12: 7 finestre
  Azione 19: 12 finestre
  Azione 21: 65 finestre
  Azione 29: 647 finestre
  Azione 3: 7 finestre
  Azione 31: 10 finestre
  Azione 36: 52 finestre
  Azione 4: 76 finestre
  Azione 41: 36 finestre
  Azione 8: 63 finestre
Kid ID 3007.0:
  Azione 10: 337 finestre
  Azione 12: 13 finestre
  Azione 31: 93 finestre
  Azione 32: 37 finestre
  Azione 41: 31 finestre
Kid ID 3008.0:
  Azione 10: 9 finestre
  Azione 16: 5 finestre
  Azione 41: 15 finestre
Kid ID 3010.0:
  Azione 10: 50 finestre
  Azione 11: 31 finestre
  Azione 19: 18 finestre
  Azione 20: 4 finestre
  Azione 3: 25 finestre
  Azione 4: 1 finestre
  Azione 6: 19 finestre
  Azione 7: 5 finestre
  Azione 8: 8 finestre
  Azione 9: 98 finestre
Kid ID 3011.0:
  Azione 10: 170 finestre
  Azione 12: 5 finestre
  Azione 16: 14 finestre
  Azione 18: 95 finestre
  Azione 19: 38 finestre
  Azione 20: 21 finestre
  Azione 21: 14 finestre
  Azione 3: 32 finestre
  Azione 4: 89 

In [50]:
# Dizionario per contare il numero totale di finestre per ogni azione
total_action_counts = {}

# Scorro tutti i bambini e le loro azioni
for actions in kid_action_counts.values(): # Per ogni bambino
    for action, count in actions.items(): # Per ogni azione
        if action not in total_action_counts: # Se l'azione non è ancora presente   
            total_action_counts[action] = 0  # Inizializza il conteggio a zero
        total_action_counts[action] += count  # Incrementa il conteggio

# Stampo il numero totale di finestre per ogni azione
print("Numero totale di finestre per ogni azione:")
for action, count in total_action_counts.items():
    print(f"Azione {action}: {count} finestre")


total_windows = sum(total_action_counts.values())
print(f"Numero totale di finestre: {total_windows}")

Numero totale di finestre per ogni azione:
Azione 10: 948 finestre
Azione 12: 25 finestre
Azione 19: 91 finestre
Azione 21: 120 finestre
Azione 29: 710 finestre
Azione 3: 75 finestre
Azione 31: 103 finestre
Azione 36: 52 finestre
Azione 4: 171 finestre
Azione 41: 438 finestre
Azione 8: 71 finestre
Azione 32: 37 finestre
Azione 16: 19 finestre
Azione 11: 31 finestre
Azione 20: 25 finestre
Azione 6: 24 finestre
Azione 7: 5 finestre
Azione 9: 98 finestre
Azione 18: 95 finestre
Numero totale di finestre: 3138


In [60]:
#filtro per ogni
#filtro per contare il numero di finestre per ogni azione
unique_actions, counts = np.unique(Y, return_counts=True)
print(unique_actions)
action_counts = dict(zip(unique_actions, counts))
print("Numero di finestre per ogni azione:")
for action, count in action_counts.items():
    print(f"Azione {action}: {count} finestre")


#split ratio  (70% nel train e 30% nel test)
split_ratio = 0.7

# Split the data
X_train = []
Y_train = []
X_test = []
Y_test = []

for action in unique_actions:

    #trovo gli indici delle finestre corrispondenti a ciascuna azione
    action_indices = np.where(Y == action)[0]
    print(action_indices)

    # Calcolo il numero di finestre da usare per il train e per il test
    num_windows = len(action_indices)
    num_train = int(num_windows * split_ratio)
    num_test = num_windows - num_train
    
    # Divido gli indici delle finestre in train e test
    train_indices = action_indices[:num_train]
    test_indices = action_indices[-num_test:]
    
    # Aggiungo le finestre al train e al test set
    X_train.append(X[train_indices])
    Y_train.append(Y[train_indices])
    X_test.append(X[test_indices])
    Y_test.append(Y[test_indices])

# Concateno tutti i dati in un unico array
X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)
X_test = np.concatenate(X_test, axis=0)
Y_test = np.concatenate(Y_test, axis=0)

print("Training set shape:", X_train.shape, Y_train.shape)
print (Y_train)
print("Test set shape:", X_test.shape, Y_test.shape)
print(Y_test)

#stampo il tipo di valore che contiene x_train e y_train(se int float ecc)
print("Tipo di X_train:", X_train.dtype)
print("Tipo di Y_train:", Y_train.dtype)

#stampo il tipo di x_train y train x test e y test
print("Tipo di X_train:", type(X_train))
print("Tipo di Y_train:", type(Y_train))
print("Tipo di X_test:", type(X_test))
print("Tipo di Y_test:", type(Y_test))

Y_train = Y_train.flatten()
Y_test = Y_test.flatten()

print("Etichette train:", Y_train)
print("Etichette test:", Y_test)

#tipo
print("Tipo di Y_train:", type(Y_train))
print("Tipo di Y_test:", type(Y_test))

[ 3  4  6  7  8  9 10 11 12 16 18 19 20 21 29 31 32 36 41]
Numero di finestre per ogni azione:
Azione 3: 75 finestre
Azione 4: 171 finestre
Azione 6: 24 finestre
Azione 7: 5 finestre
Azione 8: 71 finestre
Azione 9: 98 finestre
Azione 10: 948 finestre
Azione 11: 31 finestre
Azione 12: 25 finestre
Azione 16: 19 finestre
Azione 18: 95 finestre
Azione 19: 91 finestre
Azione 20: 25 finestre
Azione 21: 120 finestre
Azione 29: 710 finestre
Azione 31: 103 finestre
Azione 32: 37 finestre
Azione 36: 52 finestre
Azione 41: 438 finestre
[2064 2065 2066 2067 2068 2069 2070 2071 2072 2073 2074 2075 2076 2077
 2078 2079 2080 2081 2082 2083 2084 2085 2086 2087 2088 2089 2090 2091
 2092 2093 2094 2095 2096 2097 2098 2099 2100 2101 2102 2103 2104 2105
 2106 2107 2108 2109 2110 2111 2112 2113 2114 2115 2116 2117 2118 2119
 2120 2121 2122 2123 2124 2125 2126 2127 2128 2129 2130 2131 2132 2133
 2134 2135 2136 2137 2138]
[2331 2332 2333 2334 2335 2336 2337 2338 2339 2340 2341 2342 2343 2344
 2345 2346 2347 

In [66]:
# Trova tutte le etichette uniche presenti nei dati
unique_labels = np.unique(Y_train)

# Crea un dizionario che mappa ogni etichetta originale a un valore consecutivo
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}

# Stampa il dizionario per vedere il mapping
print("Mapping delle etichette:", label_mapping)

# Applica il mapping ai dataset di train e test
Y_train_mapped = np.array([label_mapping[y] for y in Y_train])
Y_test_mapped = np.array([label_mapping[y] for y in Y_test])

# Controllo finale
print("Nuove etichette train:", np.unique(Y_train_mapped))
print("Nuove etichette test:", np.unique(Y_test_mapped))


# Creo i dataset per il training e il test
train_dataset = HARDataset(X_train, Y_train_mapped)
test_dataset = HARDataset(X_test, Y_test_mapped)

 #stampo il dataset di training e di test a livello di dimensioni
print("Dataset di training:", len(train_dataset))
print("Dataset di test:", len(test_dataset))

print("Tipo di train_dataset:", type(train_dataset))
print("Tipo di test_dataset:", type(test_dataset))



# Creazione sampler pesato per il dataset di training
train_sampler = create_weighted_sampler(Y_train_mapped)



def objective(trial):
    # Definisci gli iperparametri da ottimizzare
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [4, 8, 12])

    # Crea i DataLoader con il batch_size suggerito
    #runno di nuovo il train con 5 secondi di finestra 
    train_loader = DataLoader(train_dataset, batch_size=batch_size, drop_last=True,sampler=train_sampler, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

    # Crea il modello con gli iperparametri suggeriti
    net = DeepConvLSTM(n_classes=len(unique_labels))

    # Esegui l'allenamento
    best_f1_score = train_toys.train(net, train_loader, test_loader, epochs=100, batch_size=batch_size, lr=lr)
    
    return best_f1_score

# Creazione studio Optuna ottimizza, nel senso di minimizzare la loss in 100 prove
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("Best hyperparameters: ", study.best_params)
print("Highest F1-score: ", study.best_value)

#visualizzare la storia dell'ottimizzazione effettuata da Optuna. Ci permette di vedere come la loss
# è cambiata nel corso delle diverse prove (trials) durante l'ottimizzazione.
vis.plot_optimization_history(study)

[I 2025-02-25 12:06:55,283] A new study created in memory with name: no-name-ada1c5b7-88cc-43c1-8535-fa6a3f158b6e


Mapping delle etichette: {np.int64(3): 0, np.int64(4): 1, np.int64(6): 2, np.int64(7): 3, np.int64(8): 4, np.int64(9): 5, np.int64(10): 6, np.int64(11): 7, np.int64(12): 8, np.int64(16): 9, np.int64(18): 10, np.int64(19): 11, np.int64(20): 12, np.int64(21): 13, np.int64(29): 14, np.int64(31): 15, np.int64(32): 16, np.int64(36): 17, np.int64(41): 18}
Nuove etichette train: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
Nuove etichette test: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
Dataset di training: 2186
Dataset di test: 952
Tipo di train_dataset: <class 'models.DeepConvLST_toys.HARDataset'>
Tipo di test_dataset: <class 'models.DeepConvLST_toys.HARDataset'>
Epoch: 1/100... Train Loss: 2.9491... Val Loss: 2.9158... Val Acc: 0.0326... F1-Score: 0.0331
Epoch: 2/100... Train Loss: 2.9487... Val Loss: 2.9699... Val Acc: 0.0326... F1-Score: 0.0331
Epoch: 3/100... Train Loss: 2.9475... Val Loss: 2.9676... Val Acc: 0.0326... F1-Score: 0.0331
Epoch: 4/100... Train

[I 2025-02-25 12:07:53,256] Trial 0 finished with value: 0.042787114845938376 and parameters: {'lr': 0.0006688888716088938, 'batch_size': 8}. Best is trial 0 with value: 0.042787114845938376.


Epoch: 8/100... Train Loss: 2.9450... Val Loss: 2.9728... Val Acc: 0.0431... F1-Score: 0.0428
Early stopping triggered
Epoch: 1/100... Train Loss: 2.9533... Val Loss: 2.9571... Val Acc: 0.0105... F1-Score: 0.0115
Epoch: 2/100... Train Loss: 2.9587... Val Loss: 2.8940... Val Acc: 0.3006... F1-Score: 0.3041
Epoch: 3/100... Train Loss: 2.9554... Val Loss: 2.9161... Val Acc: 0.0063... F1-Score: 0.0099
Epoch: 4/100... Train Loss: 2.9542... Val Loss: 3.0046... Val Acc: 0.0105... F1-Score: 0.0115
Epoch: 5/100... Train Loss: 2.9531... Val Loss: 2.9225... Val Acc: 0.0169... F1-Score: 0.0190
Epoch: 6/100... Train Loss: 2.9551... Val Loss: 2.9280... Val Acc: 0.0105... F1-Score: 0.0115
Epoch: 7/100... Train Loss: 2.9529... Val Loss: 2.9335... Val Acc: 0.1350... F1-Score: 0.1367
Epoch: 8/100... Train Loss: 2.9487... Val Loss: 2.9981... Val Acc: 0.0084... F1-Score: 0.0101


[I 2025-02-25 12:08:43,324] Trial 1 finished with value: 0.3040829922908537 and parameters: {'lr': 0.011392750337905402, 'batch_size': 12}. Best is trial 1 with value: 0.3040829922908537.


...

Epoch: 16/100... Train Loss: 2.9703... Val Loss: 3.0285... Val Acc: 0.0063... F1-Score: 0.0099
Early stopping triggered
Epoch: 1/100... Train Loss: 2.9486... Val Loss: 2.9337... Val Acc: 0.0105... F1-Score: 0.0115
Epoch: 2/100... Train Loss: 2.9501... Val Loss: 2.9549... Val Acc: 0.0549... F1-Score: 0.0576
Epoch: 3/100... Train Loss: 2.9467... Val Loss: 3.0096... Val Acc: 0.0549... F1-Score: 0.0551
Epoch: 4/100... Train Loss: 2.9493... Val Loss: 2.9780... Val Acc: 0.0084... F1-Score: 0.0113
Epoch: 5/100... Train Loss: 2.9472... Val Loss: 2.9581... Val Acc: 0.0084... F1-Score: 0.0113
Epoch: 6/100... Train Loss: 2.9469... Val Loss: 2.9732... Val Acc: 0.0084... F1-Score: 0.0113
Epoch: 7/100... Train Loss: 2.9480... Val Loss: 2.9693... Val Acc: 0.0295... F1-Score: 0.0322


[I 2025-02-25 12:20:07,496] Trial 9 finished with value: 0.05764362220058422 and parameters: {'lr': 0.0020597725966461393, 'batch_size': 12}. Best is trial 1 with value: 0.3040829922908537.


Epoch: 8/100... Train Loss: 2.9468... Val Loss: 2.9830... Val Acc: 0.0063... F1-Score: 0.0099
Early stopping triggered


[W 2025-02-25 12:20:19,208] Trial 10 failed with parameters: {'lr': 0.010017193532098359, 'batch_size': 4} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_5204\3829706271.py", line 51, in objective
    best_f1_score = train_toys.train(net, train_loader, test_loader, epochs=100, batch_size=batch_size, lr=lr)
  File "c:\codes\HumanActivityRecognition\HumanActivityRecognition\train_toys.py", line 78, in train
    output, val_h = net(inputs, val_h, batch_size)
  File "c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\torch\nn\modules\module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\torch\nn\modules\module.py", line 1747, in

KeyboardInterrupt: 

SUDDIVISIONE IN TRAIN E TEST PER KID_ID:
1) Stesso kid non si trova contemporaneamente sia nel train che nel test
2) Azioni bilanciate in modo che il numero totale di finestre per ogni azione rispetti (70/30 di split)
3) il train e il test contengono tutte le azioni

In [ ]:
from collections import defaultdict
import random

# Inizializzo i dizionari per contare le finestre per ogni azione nel train e nel test
train_action_counts = defaultdict(int)
test_action_counts = defaultdict(int)

# Inizializzo i set per tenere traccia dei kid_id nel train e nel test
train_kids = set()
test_kids = set()

# Calcolo il target per il numero di finestre nel train (70% del totale)
total_windows = sum(total_action_counts.values())
train_target = int(0.7 * total_windows)

# Funzione per verificare se aggiungere un kid_id al train mantiene il bilanciamento delle azioni
def can_add_to_train(kid_id): # Verifica se aggiungere un kid_id al train mantiene il bilanciamento delle azioni
    temp_train_action_counts = train_action_counts.copy()  # Copia del dizionario delle azioni nel train
    for action, count in kid_action_counts[kid_id].items(): # Per ogni azione del bambino
        temp_train_action_counts[action] += count # Incrementa il conteggio delle azioni nel train
    return all(temp_train_action_counts[action] <= train_target * (count / total_windows) for action, count in total_action_counts.items()) # Verifica se il bilanciamento è rispettato

# Funzione per assicurarsi che tutte le azioni siano nel train e nel test
def ensure_all_actions_present():
    for action in total_action_counts:
        # Se l'azione non è presente nel train, la aggiungiamo al train
        if train_action_counts[action] == 0:
            for kid_id in kid_ids:
                if action in kid_action_counts[kid_id]:
                    train_kids.add(kid_id)
                    train_action_counts[action] += kid_action_counts[kid_id][action]
                    break
        # Se l'azione non è presente nel test, la aggiungiamo al test
        if test_action_counts[action] == 0:
            for kid_id in kid_ids:
                if action in kid_action_counts[kid_id]:
                    test_kids.add(kid_id)
                    test_action_counts[action] += kid_action_counts[kid_id][action]
                    break

# Lista di kid_id da processare
kid_ids = list(kid_action_counts.keys())
random.shuffle(kid_ids)  # Shuffle per randomizzare l'ordine

# Assegno i kid_id al train o al test
for kid_id in kid_ids:
    for action, count in kid_action_counts[kid_id].items():
        # Aggiungi più finestre nel train che nel test per ogni azione
        if train_action_counts[action] < (train_target * (total_action_counts[action] / total_windows)):
            train_kids.add(kid_id)
            train_action_counts[action] += count
        else:
            test_kids.add(kid_id)
            test_action_counts[action] += count

# Assicuriamo che tutte le azioni siano presenti nel train e nel test
ensure_all_actions_present()

# Stampa i risultati
print("Train kids:", train_kids)
print("Test kids:", test_kids)

# Dettagli delle azioni nel train set
print("\nDettagli del train set:")
for action, count in train_action_counts.items():
    print(f"Azione {action}: {count} finestre")

# Numero totale di azioni nel train
total_train_actions = len(train_action_counts)
print(f"\nNumero totale di azioni nel train set: {total_train_actions}")

# Dettagli delle azioni nel test set
print("\nDettagli del test set:")
for action, count in test_action_counts.items():
    print(f"Azione {action}: {count} finestre")

# Numero totale di azioni nel test
total_test_actions = len(test_action_counts)
print(f"\nNumero totale di azioni nel test set: {total_test_actions}")


Train kids: {np.float64(3008.0), np.float64(3010.0), np.float64(3011.0), np.float64(3013.0), np.float64(3017.0), np.float64(3023.0), np.float64(3003.0), np.float64(3007.0)}
Test kids: {np.float64(3008.0), np.float64(3010.0), np.float64(3011.0), np.float64(3013.0), np.float64(3017.0), np.float64(3020.0), np.float64(3003.0), np.float64(3007.0)}

Dettagli del train set:
Azione 3: 65 finestre
Azione 10: 837 finestre
Azione 16: 19 finestre
Azione 41: 407 finestre
Azione 19: 91 finestre
Azione 4: 171 finestre
Azione 11: 31 finestre
Azione 20: 25 finestre
Azione 6: 19 finestre
Azione 7: 5 finestre
Azione 8: 71 finestre
Azione 9: 98 finestre
Azione 12: 25 finestre
Azione 21: 120 finestre
Azione 29: 647 finestre
Azione 31: 103 finestre
Azione 36: 52 finestre
Azione 18: 95 finestre
Azione 32: 37 finestre

Numero totale di azioni nel train set: 19

Dettagli del test set:
Azione 41: 31 finestre
Azione 10: 111 finestre
Azione 29: 63 finestre
Azione 6: 5 finestre
Azione 3: 10 finestre
Azione 12: 7 f

In [59]:
# Ora dividiamo effettivamente i dati in X_train, Y_train, X_test, Y_test basandoci sui kid_id
X_train = []
Y_train = []
X_test = []
Y_test = []

for kid_id in train_kids:
    for action, count in kid_action_counts[kid_id].items():
        action_indices = np.where(Y == action)[0]
        X_train.append(X[action_indices])
        Y_train.append(Y[action_indices])

for kid_id in test_kids:
    for action, count in kid_action_counts[kid_id].items():
        action_indices = np.where(Y == action)[0]
        X_test.append(X[action_indices])
        Y_test.append(Y[action_indices])

# Concateno i dati per il train e il test
X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)
X_test = np.concatenate(X_test, axis=0)
Y_test = np.concatenate(Y_test, axis=0)

# Stampa le forme finali dei dati
print("Training set shape:", X_train.shape, Y_train.shape)
print("Test set shape:", X_test.shape, Y_test.shape)

# Stampa il tipo di valore che contiene X_train e Y_train
print("Tipo di X_train:", X_train.dtype)
print("Tipo di Y_train:", Y_train.dtype)

# Stampa il tipo di X_train, Y_train, X_test e Y_test
print("Tipo di X_train:", type(X_train))
print("Tipo di Y_train:", type(Y_train))
print("Tipo di X_test:", type(X_test))
print("Tipo di Y_test:", type(Y_test))

# Appiattisco Y_train e Y_test (se necessario)
Y_train = Y_train.flatten()
Y_test = Y_test.flatten()

# Stampa le etichette finali per il train e il test
print("Etichette train:", Y_train)
print("Etichette test:", Y_test)

# Tipo finale delle etichette
print("Tipo di Y_train:", type(Y_train))
print("Tipo di Y_test:", type(Y_test))

Training set shape: (12831, 100, 13) (12831, 1)
Test set shape: (12831, 100, 13) (12831, 1)
Tipo di X_train: float64
Tipo di Y_train: int64
Tipo di X_train: <class 'numpy.ndarray'>
Tipo di Y_train: <class 'numpy.ndarray'>
Tipo di X_test: <class 'numpy.ndarray'>
Tipo di Y_test: <class 'numpy.ndarray'>
Etichette train: [10 10 10 ... 41 41 41]
Etichette test: [10 10 10 ... 41 41 41]
Tipo di Y_train: <class 'numpy.ndarray'>
Tipo di Y_test: <class 'numpy.ndarray'>
